In [0]:
spark

#####%md
#####Load the dataset setup
Upload the CSV to DBFS first: Data → Add Data → Upload File. Then run this cell to load it into a Spark DataFrame

In [0]:
df = spark.read.table("inlighnx_global_pvt.churn_prediction.wa_fn_use_c_telco_customer_churn")


##### print the schema

In [0]:

print(f"Rows: {df.count()}  |  Columns: {len(df.columns)}")
df.printSchema()

Rows: 7043  |  Columns: 21
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: long (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: long (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



#####Inspect the data EDA
Get a full picture before touching anything — missing values, duplicates, value distributions.

In [0]:
# Cell 2 — Initial inspection
from pyspark.sql import functions as F

# 2a. Preview first 5 rows
display(df.limit(5))

# 2b. Count nulls per column
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
display(null_counts)

# 2c. Check for duplicate customerIDs
dupe_count = df.count() - df.dropDuplicates(["customerID"]).count()
print(f"Duplicate rows: {dupe_count}")

# 2d. Churn distribution
display(df.groupBy("Churn").count().orderBy("count", ascending=False))

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes


customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Duplicate rows: 0


Churn,count
No,5174
Yes,1869


#####Fix Total Charges column fix
TotalCharges is loaded as a string. 11 rows have blank values (new customers with 0 tenure). Cast to float and fill blanks with 0.

In [0]:
# Cell 3 — Fix TotalCharges (string → float, blanks → 0.0)
df = df.withColumn(
    "TotalCharges",
    F.when(
        F.trim(F.col("TotalCharges")) == "",
        F.lit(0.0)
    ).otherwise(
        F.col("TotalCharges").cast("float")
    )
)

# Verify — should print 0 blanks and dtype float
print(df.schema["TotalCharges"].dataType)
df.filter(F.col("TotalCharges").isNull()).show()

DoubleType()
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract|PaperlessBilling|PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+-----

#####Handle SeniorCitizen column fix
SeniorCitizen is already an integer (0/1) but needs a readable label column for analysis and dashboards.

In [0]:
# Cell 4 — Add human-readable SeniorCitizen label
df = df.withColumn(
    "SeniorCitizenLabel",
    F.when(F.col("SeniorCitizen") == 1, "Yes").otherwise("No")
)

# Confirm distribution
display(df.groupBy("SeniorCitizenLabel").count())

SeniorCitizenLabel,count
No,5901
Yes,1142


#####Encode Churn as binary fix
Add a numeric ChurnBinary column (1 = churned, 0 = retained) needed for all ML models and aggregation queries.

In [0]:
# Cell 5 — Encode Churn as binary integer
df = df.withColumn(
    "ChurnBinary",
    F.when(F.col("Churn") == "Yes", 1).otherwise(0)
)

# Quick check
display(df.groupBy("Churn", "ChurnBinary").count())

Churn,ChurnBinary,count
No,0,5174
Yes,1,1869


#####Add tenure bands feature
Bucket continuous tenure (0–72 months) into 4 meaningful groups for analysis and dashboards.



In [0]:
# Cell 6 — Tenure bucketing
df = df.withColumn(
    "TenureGroup",
    F.when(F.col("tenure") <= 12, "0–12 months")
     .when(F.col("tenure") <= 24, "13–24 months")
     .when(F.col("tenure") <= 48, "25–48 months")
     .otherwise("49+ months")
)

display(df.groupBy("TenureGroup").count().orderBy("count", ascending=False))


TenureGroup,count
49+ months,2239
0–12 months,2186
25–48 months,1594
13–24 months,1024


#####Detect and handle outliers check
Check MonthlyCharges and TotalCharges for extreme outliers using IQR method.

In [0]:
# Cell 7 — Outlier detection on numeric columns
numeric_cols = ["MonthlyCharges", "TotalCharges", "tenure"]

for col_name in numeric_cols:
    quantiles = df.approxQuantile(col_name, [0.25, 0.75], 0.01)
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df.filter(
        (F.col(col_name) < lower) | (F.col(col_name) > upper)
    ).count()
    print(f"{col_name}: Q1={q1:.1f}, Q3={q3:.1f}, IQR bounds=[{lower:.1f}, {upper:.1f}], outliers={outliers}")

# Summary stats
display(df.select(numeric_cols).summary())

MonthlyCharges: Q1=36.2, Q3=89.5, IQR bounds=[-43.7, 169.5], outliers=0
TotalCharges: Q1=411.1, Q3=3673.6, IQR bounds=[-4482.5, 8567.3], outliers=4
tenure: Q1=9.0, Q3=55.0, IQR bounds=[-60.0, 124.0], outliers=0


summary,MonthlyCharges,TotalCharges,tenure
count,7043,7043,7043
mean,64.76169246059922,2279.7343041066433,32.37114865824223
stddev,30.09004709767847,2266.7944709124386,24.55948102309448
min,18.25,0.0,0
25%,35.5,397.0,9
50%,70.35,1393.5999755859375,29
75%,89.85,3784.0,55
max,118.75,8684.7998046875,72


#####Final validation check
Confirm the clean dataset is complete, correct, and ready for analysis.

In [0]:
print("=== CLEAN DATASET REPORT ===")
print(f"Total rows    : {df.count()}")
print(f"Total columns : {len(df.columns)}")
print(f"Columns       : {df.columns}")

# Null check across all columns
null_check = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
display(null_check)

# Churn rate
churn_rate = df.agg(
    (F.sum("ChurnBinary") / F.count("*") * 100).alias("churn_rate_pct")
).collect()[0][0]
print(f"Churn rate    : {churn_rate:.2f}%")

=== CLEAN DATASET REPORT ===
Total rows    : 7043
Total columns : 24
Columns       : ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'SeniorCitizenLabel', 'ChurnBinary', 'TenureGroup']


customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,SeniorCitizenLabel,ChurnBinary,TenureGroup
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Churn rate    : 26.54%


##### saved the clean dataset

In [0]:
df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("inlighnx_global_pvt.churn_silver.churn_clean")